# Question 1 – Search Algorithms: The Warehouse Logistics Bot

**Course:** Artificial Intelligence (ARI711S)  
**Assignment:** 1  

---

## Background

An Automated Guided Vehicle (AGV) must navigate a warehouse grid from a **Charging Station (A)** to a **Product Bin (B)**. The grid contains shelving units (walls) that the robot must route around.

We implement and compare two **informed search algorithms**:
- **Greedy Best-First Search** – uses only the heuristic `h(n)`
- **A\* Search** – uses `f(n) = g(n) + h(n)` for optimal pathfinding

---

## Problem Formulation

| Element | Description |
|---|---|
| **States** | Grid coordinates `(row, col)` |
| **Actions** | up, down, left, right |
| **Initial State** | Position of `A` |
| **Goal State** | Position of `B` |
| **Heuristic** | Euclidean distance: `h(n) = √((x1-x2)² + (y1-y2)²)` |

## Step 1 – Imports

In [ ]:
import heapq
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## Step 2 – Node Class

Each `Node` represents one position in the search tree. It stores:
- `state` — the `(row, col)` grid coordinate
- `parent` — the node we came from (used to reconstruct the path)
- `action` — the direction we moved to reach this node
- `g` — the path cost from the start to this node (number of steps)

In [ ]:
class Node:
    """
    Represents a single state in the search tree.

    Attributes:
        state  : (row, col) grid coordinate
        parent : the Node we arrived from (None for the start node)
        action : direction string taken to reach this node
        g      : path cost from the start node to this node
    """

    def __init__(self, state, parent, action, g):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g  # g(n) — cost so far

    def __lt__(self, other):
        # Needed so heapq can break ties without error
        return self.g < other.g

print("Node class defined.")

## Step 3 – Warehouse Class

The `Warehouse` class:
1. **Reads and parses** the `.txt` file into a 2-D grid
2. **Locates** the coordinates of `A` (start) and `B` (goal)
3. **Stores** all wall positions (shelving units)
4. Provides a `neighbors(state)` method returning valid adjacent aisles
5. Computes the **Euclidean distance heuristic** `h(n)`
6. Implements `solve(algorithm)` using a **Priority Queue** frontier
7. Exports a colour-coded PNG via `visualize()`

In [ ]:
class Warehouse:
    """
    Models the warehouse grid and exposes two informed search algorithms:
    Greedy Best-First Search and A* Search.

    Grid legend (in the .txt file):
        #   – shelving unit / wall (impassable)
        A   – charging station (start)
        B   – product bin (goal)
        ' ' – open aisle (passable)
    """

    def __init__(self, filename: str):
        self.walls = set()
        self.start = None
        self.goal = None
        self._parse(filename)

    # ── File Parsing ──────────────────────────────────────────────────────────
    def _parse(self, filename: str):
        """Read the .txt file and populate walls, start, and goal."""
        with open(filename) as f:
            lines = f.read().splitlines()

        self.height = len(lines)
        self.width = max(len(line) for line in lines)

        for r, line in enumerate(lines):
            for c, ch in enumerate(line):
                if ch == "#":
                    self.walls.add((r, c))
                elif ch == "A":
                    self.start = (r, c)
                elif ch == "B":
                    self.goal = (r, c)

        if self.start is None or self.goal is None:
            raise ValueError("warehouse.txt must contain both 'A' (start) and 'B' (goal).")

        print(f"Parsed warehouse: {self.height} rows x {self.width} cols")
        print(f"  Start (A) : {self.start}")
        print(f"  Goal  (B) : {self.goal}")
        print(f"  Walls     : {len(self.walls)} cells")

    # ── Neighbour Expansion ───────────────────────────────────────────────────
    def neighbors(self, state):
        """
        Returns a list of (action, new_state) for all valid moves from state.
        A move is valid when the target cell is inside the grid and not a wall.
        """
        row, col = state
        candidates = [
            ("up",    (row - 1, col)),
            ("down",  (row + 1, col)),
            ("left",  (row,     col - 1)),
            ("right", (row,     col + 1)),
        ]
        return [
            (action, s)
            for action, s in candidates
            if 0 <= s[0] < self.height
            and 0 <= s[1] < self.width
            and s not in self.walls
        ]

    # ── Heuristic: Euclidean Distance ─────────────────────────────────────────
    def _heuristic(self, state):
        """
        h(n) = Euclidean distance from state to the goal.
        h(n) = sqrt((x1 - x2)^2 + (y1 - y2)^2)
        """
        x1, y1 = state
        x2, y2 = self.goal
        return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)

    # ── Solve ─────────────────────────────────────────────────────────────────
    def solve(self, algorithm: str = "astar"):
        """
        Finds the path from start (A) to goal (B).

        Parameters
        ----------
        algorithm : 'greedy'  ->  priority = h(n)          [Greedy Best-First]
                    'astar'   ->  priority = g(n) + h(n)   [A* Search]

        Returns
        -------
        path     : list of (row, col) from start to goal (inclusive)
        explored : set of all states that were expanded during search
        """
        algorithm = algorithm.lower()
        if algorithm not in ("greedy", "astar"):
            raise ValueError("algorithm must be 'greedy' or 'astar'")

        start_node = Node(state=self.start, parent=None, action=None, g=0)

        # Priority queue entries: (priority, tie-breaking counter, node)
        counter = 0
        frontier = []
        h0 = self._heuristic(self.start)
        priority = h0 if algorithm == "greedy" else 0 + h0
        heapq.heappush(frontier, (priority, counter, start_node))

        explored = set()          # states already expanded
        frontier_states = {self.start}  # fast membership test for frontier

        while frontier:
            _, _, node = heapq.heappop(frontier)
            frontier_states.discard(node.state)

            # ── Goal test ──────────────────────────
            if node.state == self.goal:
                path = self._reconstruct_path(node)
                print(f"\n[{algorithm.upper()}] Solution found!")
                print(f"  Path length     : {len(path)} steps")
                print(f"  States explored : {len(explored)}")
                return path, explored

            # ── Skip if already visited ────────────
            if node.state in explored:
                continue
            explored.add(node.state)

            # ── Expand neighbours ──────────────────
            for action, state in self.neighbors(node.state):
                if state in explored or state in frontier_states:
                    continue
                new_g = node.g + 1
                h = self._heuristic(state)
                # Greedy: priority = h(n)
                # A*    : priority = g(n) + h(n)
                priority = h if algorithm == "greedy" else new_g + h
                counter += 1
                child = Node(state=state, parent=node, action=action, g=new_g)
                heapq.heappush(frontier, (priority, counter, child))
                frontier_states.add(state)

        print(f"[{algorithm.upper()}] No solution found.")
        return None, explored

    # ── Path Reconstruction ───────────────────────────────────────────────────
    def _reconstruct_path(self, node):
        """Walk parent pointers back to start, then reverse."""
        path = []
        while node is not None:
            path.append(node.state)
            node = node.parent
        path.reverse()
        return path

    # ── Visualisation ─────────────────────────────────────────────────────────
    def visualize(self, path, explored, filename="warehouse_path.png", algorithm="astar"):
        """
        Exports a colour-coded PNG of the warehouse showing:
          - Shelving units (walls)
          - States explored but not on the final path
          - The robot's optimal path
          - Start (A) and Goal (B) markers
        """
        # Numeric grid: 0=open, 1=wall, 2=explored, 3=path, 4=start, 5=goal
        grid = np.zeros((self.height, self.width), dtype=int)

        for (r, c) in self.walls:
            grid[r, c] = 1
        if explored:
            for (r, c) in explored:
                if grid[r, c] == 0:
                    grid[r, c] = 2
        if path:
            for (r, c) in path:
                grid[r, c] = 3
        if self.start:
            grid[self.start[0], self.start[1]] = 4
        if self.goal:
            grid[self.goal[0], self.goal[1]] = 5

        cmap = plt.cm.colors.ListedColormap([
            "white",    # 0 – open aisle
            "#2b2b2b",  # 1 – wall
            "#aed6f1",  # 2 – explored
            "#f9e547",  # 3 – optimal path
            "#27ae60",  # 4 – start A
            "#e74c3c",  # 5 – goal  B
        ])

        fig, ax = plt.subplots(figsize=(max(8, self.width * 0.5),
                                        max(6, self.height * 0.5)))
        ax.imshow(grid, cmap=cmap, vmin=0, vmax=5, interpolation="nearest")

        # Labels
        ax.text(self.start[1], self.start[0], "A", ha="center", va="center",
                fontsize=9, fontweight="bold", color="white")
        ax.text(self.goal[1], self.goal[0], "B", ha="center", va="center",
                fontsize=9, fontweight="bold", color="white")

        # Grid lines
        ax.set_xticks(np.arange(-0.5, self.width, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, self.height, 1), minor=True)
        ax.grid(which="minor", color="grey", linewidth=0.3)
        ax.tick_params(which="both", bottom=False, left=False,
                       labelbottom=False, labelleft=False)

        # Legend
        patches = [
            mpatches.Patch(color="#27ae60", label="Start (A)"),
            mpatches.Patch(color="#e74c3c", label="Goal (B)"),
            mpatches.Patch(color="#f9e547", label="Optimal path"),
            mpatches.Patch(color="#aed6f1", label="Explored (not on path)"),
            mpatches.Patch(color="#2b2b2b", label="Shelving unit (wall)"),
            mpatches.Patch(color="white",   label="Open aisle", ec="grey"),
        ]
        ax.legend(handles=patches, loc="upper right", fontsize=7, framealpha=0.9)

        steps = len(path) if path else 'N/A'
        ax.set_title(
            f"Warehouse AGV Navigation – {algorithm.upper()}\n"
            f"Path length: {steps} steps  |  States explored: {len(explored)}",
            fontsize=10
        )

        plt.tight_layout()
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Image saved -> {filename}")


print("Warehouse class defined.")

## Step 4 – Load the Warehouse

We parse `warehouse.txt`. The file uses:
- `#` for shelving units (walls)
- `A` for the charging station (start)
- `B` for the product bin (goal)
- spaces for open aisles

In [ ]:
wh = Warehouse("warehouse.txt")

## Step 5 – Greedy Best-First Search

**Priority = h(n)** (Euclidean distance to goal only).  
Greedy always expands the node that *looks* closest to the goal — it is fast but **not guaranteed to find the shortest path**.

In [ ]:
greedy_path, greedy_explored = wh.solve(algorithm="greedy")
wh.visualize(greedy_path, greedy_explored,
             filename="warehouse_path_greedy.png",
             algorithm="greedy")

## Step 6 – A* Search

**Priority = g(n) + h(n)** (actual cost so far + estimated cost to goal).  
A\* is **optimal and complete** — it always finds the shortest path, provided the heuristic is admissible (Euclidean distance never overestimates).

In [ ]:
astar_path, astar_explored = wh.solve(algorithm="astar")
wh.visualize(astar_path, astar_explored,
             filename="warehouse_path.png",
             algorithm="astar")

## Step 7 – Comparison

We compare both algorithms on **path length** and **number of states explored**.

In [ ]:
print(f"{'Algorithm':<12} {'Path Length':>14} {'States Explored':>17}")
print("-" * 45)

gl = len(greedy_path) if greedy_path else "No solution"
ge = len(greedy_explored)
al = len(astar_path) if astar_path else "No solution"
ae = len(astar_explored)

print(f"{'Greedy':<12} {str(gl):>14} {str(ge):>17}")
print(f"{'A*':<12} {str(al):>14} {str(ae):>17}")

print("\nAnalysis:")
print("- Greedy is typically faster (fewer nodes explored) but may find a longer path.")
print("- A* explores more nodes but guarantees the shortest possible path.")
print("- The Euclidean heuristic is admissible, so A* is optimal.")

## Step 8 – Test Cases

We verify correct behaviour on edge cases.

In [ ]:
import tempfile, os

def make_tmp_warehouse(content):
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False)
    tmp.write(content)
    tmp.close()
    return tmp.name

# ── Test 1: Straight-line path (no walls) ──────────────────────────────────
print("TEST 1: Straight-line, no walls")
f = make_tmp_warehouse("A    B")
wt = Warehouse(f)
path, _ = wt.solve("astar")
assert path is not None and path[0] == wt.start and path[-1] == wt.goal
assert len(path) == 6  # A + 4 spaces + B
print("  PASSED — path length:", len(path))
os.unlink(f)

# ── Test 2: Blocked goal (no solution) ────────────────────────────────────
print("\nTEST 2: Goal is completely blocked")
blocked = "A   #\n#####\n####B"
f = make_tmp_warehouse(blocked)
wt = Warehouse(f)
path, _ = wt.solve("astar")
assert path is None
print("  PASSED — correctly returned None (no solution)")
os.unlink(f)

# ── Test 3: A and B are adjacent ──────────────────────────────────────────
print("\nTEST 3: A and B adjacent")
f = make_tmp_warehouse("AB")
wt = Warehouse(f)
path, _ = wt.solve("astar")
assert path is not None and len(path) == 2
print("  PASSED — path length:", len(path))
os.unlink(f)

# ── Test 4: Greedy vs A* path length comparison ───────────────────────────
print("\nTEST 4: Greedy path length >= A* path length")
wt2 = Warehouse(make_tmp_warehouse(
    "##########\n"
    "#A  #    #\n"
    "#   #  B #\n"
    "#   #    #\n"
    "##########"
))
g_path, _ = wt2.solve("greedy")
a_path, _ = wt2.solve("astar")
if g_path and a_path:
    assert len(a_path) <= len(g_path), "A* must be at least as short as Greedy"
    print(f"  PASSED — Greedy: {len(g_path)}, A*: {len(a_path)}")

print("\nAll tests passed!")

## Summary

| Component | Implementation |
|---|---|
| `Node` class | Stores `state`, `parent`, `action`, `g(n)` |
| `Warehouse._parse()` | Reads `.txt`, finds A/B, records walls |
| `Warehouse.neighbors()` | Returns valid up/down/left/right moves |
| `Warehouse._heuristic()` | Euclidean distance to goal |
| `Warehouse.solve('greedy')` | Priority queue on `h(n)` |
| `Warehouse.solve('astar')` | Priority queue on `g(n) + h(n)` |
| `Warehouse.visualize()` | Exports `warehouse_path.png` with distinct colours |

**Key insight:** A\* guarantees the optimal (shortest) path because the Euclidean distance heuristic is *admissible* — it never overestimates the true remaining cost. Greedy may be faster in terms of nodes explored, but it can produce a suboptimal path.